In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

df = pd.read_csv("data/E2.csv")

# keep what we need, including the date
matches = df[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]].copy()

# parse the date — football-data uses day/month/year
matches["Date"] = pd.to_datetime(matches["Date"], dayfirst=True)

# sort oldest to newest
matches = matches.sort_values("Date").reset_index(drop=True)

print(matches.shape)
print("First match:", matches["Date"].min())
print("Last match: ", matches["Date"].max())
matches.head()

(552, 5)
First match: 2024-08-09 00:00:00
Last match:  2025-05-03 00:00:00


,Date,HomeTeam,AwayTeam,FTHG,FTAG
0,2024-08-09,Barnsley,Mansfield,1,2
1,2024-08-10,Wrexham,Wycombe,3,2
2,2024-08-10,Wigan,Charlton,0,1
3,2024-08-10,Stockport,Cambridge,2,0
4,2024-08-10,Peterboro,Huddersfield,0,2


In [3]:
# split point: first 70% of matches (by date) for training, last 30% for testing
split_idx = int(len(matches) * 0.70)

train = matches.iloc[:split_idx].copy()
test = matches.iloc[split_idx:].copy()

print(f"Training matches: {len(train)}  ({train['Date'].min().date()} to {train['Date'].max().date()})")
print(f"Testing matches:  {len(test)}  ({test['Date'].min().date()} to {test['Date'].max().date()})")

Training matches: 386  (2024-08-09 to 2025-02-25)
Testing matches:  166  (2025-02-25 to 2025-05-03)


In [5]:
avg_home_goals = train["FTHG"].mean()
avg_away_goals = train["FTAG"].mean()

home = train.groupby("HomeTeam").agg(
    home_scored=("FTHG", "mean"),
    home_conceded=("FTAG", "mean")
)
away = train.groupby("AwayTeam").agg(
    away_scored=("FTAG", "mean"),
    away_conceded=("FTHG", "mean")
)
strength = home.join(away)

print("Teams with strength data:", len(strength))
print("Any missing values?")
print(strength.isnull().sum())
strength.head()

Teams with strength data: 24
Any missing values?
home_scored      0
home_conceded    0
away_scored      0
away_conceded    0
dtype: int64


,home_scored,home_conceded,away_scored,away_conceded
HomeTeam,,,,
Barnsley,1.312500,1.437500,1.437500,1.250000
Birmingham,1.812500,0.437500,1.600000,0.800000
Blackpool,1.466667,1.333333,1.529412,1.470588
Bolton,1.823529,1.529412,1.400000,1.533333
Bristol Rvs,1.466667,1.266667,0.647059,1.823529


In [7]:
def predict_match(home_team, away_team):
    # expected goals (same logic as session 3, now using train-derived strengths)
    home_exp = (strength.loc[home_team, "home_scored"]
                * strength.loc[away_team, "away_conceded"]
                / avg_home_goals)
    away_exp = (strength.loc[away_team, "away_scored"]
                * strength.loc[home_team, "home_conceded"]
                / avg_away_goals)

    # combine Poisson distributions into match outcome probabilities
    max_goals = 10
    home_win, draw, away_win = 0.0, 0.0, 0.0
    for hg in range(max_goals):
        for ag in range(max_goals):
            p = poisson.pmf(hg, home_exp) * poisson.pmf(ag, away_exp)
            if hg > ag:
                home_win += p
            elif hg == ag:
                draw += p
            else:
                away_win += p
    return home_win, draw, away_win

# test the function on one fixture
predict_match("Birmingham", "Burton")

(0.7116197105066013, 0.22034989345708642, 0.06801991478147958)

In [9]:
preds = []
for _, row in test.iterrows():
    h, a = row["HomeTeam"], row["AwayTeam"]
    # skip if either team somehow isn't in the training strengths
    if h not in strength.index or a not in strength.index:
        continue
    hw, dr, aw = predict_match(h, a)
    # the model's single best guess = highest-probability outcome
    if hw >= dr and hw >= aw:
        pick = "H"
    elif aw >= hw and aw >= dr:
        pick = "A"
    else:
        pick = "D"
    # what actually happened
    actual = "H" if row["FTHG"] > row["FTAG"] else ("A" if row["FTHG"] < row["FTAG"] else "D")
    preds.append({"home": h, "away": a, "pred": pick,
                  "actual": actual, "p_home": hw, "p_draw": dr, "p_away": aw})

results = pd.DataFrame(preds)
print(f"Predicted {len(results)} matches")
results.head(10)

Predicted 166 matches


,home,away,pred,actual,p_home,p_draw,p_away
0,Northampton,Barnsley,A,A,0.206536,0.237319,0.556132
1,Wigan,Reading,H,A,0.389746,0.283105,0.327149
2,Stockport,Blackpool,H,H,0.482503,0.232681,0.284798
3,Stevenage,Huddersfield,A,A,0.229207,0.267399,0.503391
4,Peterboro,Shrewsbury,H,H,0.557849,0.208693,0.233386
5,Exeter,Northampton,H,D,0.438499,0.245259,0.316233
6,Leyton Orient,Charlton,H,A,0.500182,0.284061,0.215755
7,Burton,Mansfield,A,D,0.264538,0.222362,0.513065
8,Bristol Rvs,Rotherham,H,A,0.517942,0.278480,0.203577
9,Barnsley,Lincoln,A,H,0.314748,0.265872,0.419378


In [11]:
# overall accuracy
correct = (results["pred"] == results["actual"]).sum()
accuracy = correct / len(results)
print(f"Model accuracy: {correct}/{len(results)} = {accuracy:.3f}")

# the baseline to beat: always predict Home
home_rate = (results["actual"] == "H").mean()
print(f"Baseline (always pick Home): {home_rate:.3f}")

# breakdown: what did the model predict, and how did each fare?
print("\nWhat the model predicted:")
print(results["pred"].value_counts())

print("\nActual outcomes in test set:")
print(results["actual"].value_counts())

Model accuracy: 85/166 = 0.512
Baseline (always pick Home): 0.404

What the model predicted:
pred
H    104
A     61
D      1
Name: count, dtype: int64

Actual outcomes in test set:
actual
H    67
A    60
D    39
Name: count, dtype: int64
